# Métriques du run language FineWeb

Run lancé le **5 novembre 2026** : `language96m_fineweb_8h_BsjrP3`.
Lecture de `log.txt` uniquement : aucun checkpoint ni GPU nécessaire.
Exécuter toutes les cellules pour actualiser les courbes. Dépendances : `numpy`, `matplotlib`.


In [ ]:
%matplotlib inline
from pathlib import Path
import re
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# Fonctionne depuis la racine du dépôt ou depuis notebooks/.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'runs').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Ouvrir le notebook depuis le dépôt contenant runs/.")
RUN_DIR = ROOT / 'runs' / 'language96m_fineweb_8h_BsjrP3'
SMOOTHING = 250  # Moyenne glissante, en étapes ; 1 désactive le lissage.
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.grid': True, 'grid.alpha': 0.25})
print(f'Run : {RUN_DIR}')

## Chargement du journal

Les évaluations interrompues (`val_loss None`) sont ignorées. En cas de reprise produisant plusieurs valeurs pour une même étape, la dernière valeur est conservée. Les lignes non reconnues sont signalées.

In [ ]:
number = r'([-+]?\d+(?:\.\d*)?(?:[eE][-+]?\d+)?)'
train_pattern = re.compile(r'step\s+(\d+)\s*\| loss ' + number + r'\s*\| lr ' + number + r'\s*\| norm ' + number + r'\s*\| tok/sec ' + number)
val_pattern = re.compile(r'step\s+(\d+)\s*\| val_loss\s+(\S+)')
train_rows, val_rows, ignored = {}, {}, []
for line_no, line in enumerate((RUN_DIR / 'log.txt').read_text().splitlines(), 1):
    if match := train_pattern.fullmatch(line.strip()):
        train_rows[int(match[1])] = tuple(float(x) for x in match.groups()[1:])
    elif match := val_pattern.fullmatch(line.strip()):
        if match[2] != 'None':
            value = float(match[2])
            if np.isfinite(value):
                val_rows[int(match[1])] = value
    elif line.strip():
        ignored.append(line_no)
if not train_rows:
    raise ValueError('Aucune métrique d’entraînement trouvée dans log.txt.')
steps = np.array(sorted(train_rows))
loss, lr, norm, throughput = np.array([train_rows[s] for s in steps]).T
val_steps = np.array(sorted(val_rows), dtype=int)
val_loss = np.array([val_rows[s] for s in val_steps])

def smooth(values):
    width = min(max(1, int(SMOOTHING)), len(values))
    cumulative = np.r_[0.0, np.cumsum(values)]
    end = np.arange(1, len(values) + 1)
    start = np.maximum(0, end - width)
    return (cumulative[end] - cumulative[start]) / (end - start)

print(f'{len(steps):,} étapes — de {steps[0]} à {steps[-1]} ; {len(val_steps)} évaluations.')
if ignored:
    print(f'{len(ignored)} lignes ignorées (premières : {ignored[:10]}).')
if np.any(np.diff(steps) != 1):
    print('Attention : le journal présente des étapes manquantes.')

In [ ]:
rows = [
    ('Dernière étape', f'{steps[-1]:,}'),
    ('Loss train finale (brute)', f'{loss[-1]:.4f}'),
    ('Loss train finale (lissée)', f'{smooth(loss)[-1]:.4f}'),
    ('Learning rate final', f'{lr[-1]:.3e}'),
    ('Norme du gradient finale', f'{norm[-1]:.4f}'),
    ('Débit médian', f'{np.median(throughput):,.0f} tokens/s'),
    ('Étapes avec norme > 1', f'{100 * np.mean(norm > 1):.1f} %'),
]
if len(val_steps):
    best = np.argmin(val_loss)
    rows += [
        ('Dernière loss validation', f'{val_loss[-1]:.4f} (étape {val_steps[-1]:,})'),
        ('Meilleure loss validation', f'{val_loss[best]:.4f} (étape {val_steps[best]:,})'),
        ('Dernière perplexité validation', f'{np.exp(val_loss[-1]):.2f}'),
    ]
display(Markdown('| Mesure | Valeur |\n|---|---|\n' + '\n'.join(f'| {k} | {v} |' for k, v in rows)))

## Pertes et perplexité

La perplexité est `exp(loss)` pour la cross-entropie en logarithme naturel. Son axe est logarithmique pour rendre le début du run lisible. La loss train est mesurée pendant les mises à jour ; la validation est évaluée séparément. Leur comparaison reste indicative.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(steps, loss, alpha=0.18, color='tab:blue', label='Train brute')
axes[0].plot(steps, smooth(loss), color='tab:blue', label=f'Train lissée ({SMOOTHING} étapes)')
axes[1].plot(steps, np.exp(smooth(loss)), label='Train (exp de la loss lissée)')
if len(val_steps):
    axes[0].plot(val_steps, val_loss, color='tab:orange', label='Validation')
    axes[1].plot(val_steps, np.exp(val_loss), color='tab:orange', label='Validation')
axes[0].set(title='Cross-entropie', ylabel='Loss')
axes[1].set(title='Perplexité', ylabel='Perplexité', yscale='log')
for ax in axes:
    ax.set_xlabel('Étape'); ax.legend()
fig.tight_layout()
plt.show()

## Optimisation et débit

La norme enregistrée par `clip_grad_norm_` est celle **avant clipping**, dont le seuil est 1 dans le code du run. Le débit mesure uniquement le temps d’une étape d’entraînement : il exclut les validations et sauvegardes et ne permet pas de retrouver la durée totale du run.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)
axes[0].plot(steps, lr)
axes[0].set(title="Taux d’apprentissage", ylabel='Learning rate')
axes[0].ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
axes[1].plot(steps, norm, alpha=0.2, color='tab:purple', label='Brute')
axes[1].plot(steps, smooth(norm), color='tab:purple', label='Lissée')
axes[1].axhline(1.0, color='tab:red', linestyle='--', label='Seuil de clipping')
axes[1].set(title='Norme du gradient avant clipping', ylabel='Norme', yscale='log')
axes[1].legend()
axes[2].plot(steps, throughput, alpha=0.25, color='tab:green', label='Brut')
axes[2].plot(steps, smooth(throughput), color='tab:green', label='Lissé')
axes[2].set(title='Débit d’entraînement', ylabel='Tokens/s', xlabel='Étape')
axes[2].legend()
fig.tight_layout()
plt.show()

## Lecture des oscillations

Sur ce journal, la validation passe de **6,30** (étape 250) à **3,48** (48 250), avec un minimum de **3,470** à 48 000. La loss train des 1 000 dernières étapes vaut **3,442 ± 0,184** (moyenne ± écart-type entre étapes). Les deux mesures restent proches ; on ne voit pas de divergence durable.

- Chaque loss train moyenne **32 768 tokens**, issus de blocs consécutifs, sans mélange dans le chargeur. Ces tokens ne sont pas indépendants : les documents et leur difficulté changent entre lots. C’est une explication plausible du bruit, sans pouvoir attribuer chaque pic sans examiner les données correspondantes.
- La validation moyenne **20 micro-lots de 4 × 1 024 tokens**, soit **81 920 tokens**, et repart du même début de validation à chaque évaluation. Sa courbe est donc naturellement plus stable. Elle ne couvre pas les 10 millions de tokens de validation du manifeste.
- Quelques pics sont plus marqués : loss **5,38 puis 5,42** aux étapes 47 647–47 648. Ils sont transitoires ; la validation suivante reste proche de 3,48. Leur cause précise n’est pas établie par le journal seul.
- La norme est mesurée **avant clipping** : dépasser 1 est attendu. Le clipping protège la mise à jour, mais ne supprime pas la variabilité de la loss déjà calculée.

Le journal contient 48 354 étapes distinctes et toutes les métriques train sont finies. Ces constats sont rassurants, sans constituer une vérification complète du modèle ou du dataset.

## Génération : cinq continuations de 50 tokens

Le modèle génère exactement **50 nouveaux tokens** pour chacune des cinq copies du prompt. Échantillonnage aléatoire avec température `0.8`, `top_k=50` et graine fixe pour pouvoir reproduire l’essai dans le même environnement.

Le checkpoint doit contenir `config` et `model`, comme ceux sauvegardés par `src/train_language.py`. Par défaut, on utilise `last.pt`, puis le dernier `model_*.pt`. Si les poids sont encore sur la machine d’entraînement, les copier dans le dossier du run ou renseigner `CHECKPOINT` ci-dessous. La cellule utilise CUDA si disponible, sinon le CPU (plus lent).

In [ ]:
import sys
import torch

# Remplacer par Path('/chemin/vers/model_048250.pt') si nécessaire.
CHECKPOINT = RUN_DIR / 'last.pt'
PROMPT = "Hello, I'm a LLM, and I"
TEMPERATURE = 0.8
TOP_K = 50
SEED = 42

if not CHECKPOINT.is_file():
    candidates = sorted(RUN_DIR.glob('model_*.pt')) if CHECKPOINT == RUN_DIR / 'last.pt' else []
    if candidates:
        CHECKPOINT = candidates[-1]

if not CHECKPOINT.is_file():
    print(f"Checkpoint absent : {CHECKPOINT}\n"
          "Copier les poids du run ici ou modifier CHECKPOINT, puis réexécuter cette cellule.")
else:
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    from src.models.language import GPT, GPTConfig
    from src.models.language.tokenizer import Tokenizer

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
    model = GPT(GPTConfig(**checkpoint['config']))
    model.load_state_dict(checkpoint['model'], strict=True)
    checkpoint_step = checkpoint.get('step', '?')
    del checkpoint  # Ne pas conserver l’état de l’optimiseur en mémoire.
    model = model.to(device).eval()
    tokenizer = Tokenizer()
    if model.config.vocab_size != tokenizer.vocab_size:
        raise ValueError('Le vocabulaire du checkpoint ne correspond pas au tokenizer GPT-2.')
    prompt_ids = tokenizer.encode(PROMPT)
    inputs = torch.tensor([prompt_ids] * 5, dtype=torch.long, device=device)
    torch.manual_seed(SEED)
    if device.type == 'cuda':
        torch.cuda.manual_seed_all(SEED)
    print(f'Checkpoint : {CHECKPOINT.name} | étape {checkpoint_step} | {device}')
    with torch.inference_mode():
        samples = model.generate(inputs, max_new_tokens=50,
                                 temperature=TEMPERATURE, top_k=TOP_K).cpu().tolist()
    for i, token_ids in enumerate(samples, 1):
        continuation = tokenizer.decode(token_ids[len(prompt_ids):])
        print(f"\n{'=' * 20} Échantillon {i} — 50 nouveaux tokens {'=' * 20}")
        print(PROMPT + continuation)
